In [2]:
!pip install torch transformers scikit-learn pandas pyarrow tqdm datasets -q

In [5]:
import os
os.makedirs('/content/data/splits', exist_ok=True)
os.makedirs('/content/results', exist_ok=True)
print("Folders created")

Folders created


In [13]:
import os
os.makedirs('data/splits', exist_ok=True)

os.rename('/content/val_dedup.parquet', '/content/data/splits/val_dedup.parquet')

print("Files moved")

Files moved


In [14]:
with open('train_hybrid.py', 'r') as f:
    content = f.read()
content = content.replace('../../data/splits/', 'data/splits/')
with open('train_hybrid.py', 'w') as f:
    f.write(content)
print("Path fixed")

Path fixed


In [15]:
with open('train_hybrid.py', 'r') as f:
    content = f.read()

content = content.replace('train_sanitized.parquet', 'train_dedup.parquet')
content = content.replace('test_sanitized.parquet', 'test_dedup.parquet')
content = content.replace('val_sanitized.parquet', 'val_dedup.parquet')

with open('train_hybrid.py', 'w') as f:
    f.write(content)

print("Fixed. Running now...")

Fixed. Running now...


In [17]:
!python train_hybrid.py

Loading datasets...
Loading tokenizer...
Tokenizing datasets...
Map: 100% 2000/2000 [00:00<00:00, 3461.30 examples/s]
Map: 100% 500/500 [00:00<00:00, 3590.63 examples/s]
Loading hybrid model...
Loading weights: 100% 199/199 [00:00<00:00, 1205.54it/s, Materializing param=pooler.dense.weight]
Loading weights: 100% 197/197 [00:00<00:00, 974.26it/s, Materializing param=encoder.layer.11.output.dense.weight] 
RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
poole

In [18]:
# Replace the training section with a simple PyTorch loop
training_code = '''
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, confusion_matrix
import json
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

print("Loading data...")
train_df = pd.read_parquet("data/splits/train_dedup.parquet")
test_df  = pd.read_parquet("data/splits/test_dedup.parquet")
print(f"Train: {len(train_df):,} | Test: {len(test_df):,}")

class CodeDataset(Dataset):
    def __init__(self, df, tok_cb, tok_gcb):
        self.codes   = df["code"].tolist()
        self.labels  = df["label"].tolist()
        self.tok_cb  = tok_cb
        self.tok_gcb = tok_gcb
    def __len__(self): return len(self.codes)
    def __getitem__(self, idx):
        code = self.codes[idx]
        enc_cb  = self.tok_cb(code,  max_length=256, padding="max_length", truncation=True, return_tensors="pt")
        enc_gcb = self.tok_gcb(code, max_length=256, padding="max_length", truncation=True, return_tensors="pt")
        return {
            "cb_ids":   enc_cb["input_ids"].squeeze(),
            "cb_mask":  enc_cb["attention_mask"].squeeze(),
            "gcb_ids":  enc_gcb["input_ids"].squeeze(),
            "gcb_mask": enc_gcb["attention_mask"].squeeze(),
            "label":    torch.tensor(self.labels[idx], dtype=torch.long)
        }

class HybridModel(nn.Module):
    def __init__(self, cb, gcb):
        super().__init__()
        self.cb  = cb
        self.gcb = gcb
        for p in self.cb.parameters():  p.requires_grad = False
        for p in self.gcb.parameters(): p.requires_grad = False
        self.head = nn.Sequential(
            nn.Linear(768*2, 512), nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, 2)
        )
    def forward(self, cb_ids, cb_mask, gcb_ids, gcb_mask):
        cb_out  = self.cb(input_ids=cb_ids,   attention_mask=cb_mask).last_hidden_state[:,0,:]
        gcb_out = self.gcb(input_ids=gcb_ids, attention_mask=gcb_mask).last_hidden_state[:,0,:]
        return self.head(torch.cat([cb_out, gcb_out], dim=1))

print("Loading tokenizers...")
from transformers import AutoTokenizer, AutoModel
tok_cb  = AutoTokenizer.from_pretrained("microsoft/codebert-base")
tok_gcb = AutoTokenizer.from_pretrained("microsoft/graphcodebert-base")
cb  = AutoModel.from_pretrained("microsoft/codebert-base")
gcb = AutoModel.from_pretrained("microsoft/graphcodebert-base")

train_loader = DataLoader(CodeDataset(train_df, tok_cb, tok_gcb), batch_size=16, shuffle=True)
test_loader  = DataLoader(CodeDataset(test_df,  tok_cb, tok_gcb), batch_size=16, shuffle=False)

model     = HybridModel(cb, gcb).to(DEVICE)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-4)
criterion = nn.CrossEntropyLoss()

print("Training...")
for epoch in range(3):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/3"):
        cb_ids   = batch["cb_ids"].to(DEVICE)
        cb_mask  = batch["cb_mask"].to(DEVICE)
        gcb_ids  = batch["gcb_ids"].to(DEVICE)
        gcb_mask = batch["gcb_mask"].to(DEVICE)
        labels   = batch["label"].to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(cb_ids, cb_mask, gcb_ids, gcb_mask), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} loss: {total_loss/len(train_loader):.4f}")

print("Evaluating...")
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        cb_ids   = batch["cb_ids"].to(DEVICE)
        cb_mask  = batch["cb_mask"].to(DEVICE)
        gcb_ids  = batch["gcb_ids"].to(DEVICE)
        gcb_mask = batch["gcb_mask"].to(DEVICE)
        labels   = batch["label"].to(DEVICE)
        preds    = torch.argmax(model(cb_ids, cb_mask, gcb_ids, gcb_mask), dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, confusion_matrix
import json, os

acc  = accuracy_score(all_labels, all_preds)
f1   = f1_score(all_labels, all_preds, zero_division=0)
prec = precision_score(all_labels, all_preds, zero_division=0)
rec  = recall_score(all_labels, all_preds, zero_division=0)
cm   = confusion_matrix(all_labels, all_preds)

print(f"Accuracy:  {acc:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")

os.makedirs("results", exist_ok=True)
results = {"model":"hybrid","accuracy":round(acc,4),"f1":round(f1,4),"precision":round(prec,4),"recall":round(rec,4),"confusion_matrix":cm.tolist()}
with open("results/hybrid_evaluation.json","w") as f:
    json.dump(results, f, indent=2)
print("Saved to results/hybrid_evaluation.json")
'''

with open('train_hybrid_simple.py', 'w') as f:
    f.write(training_code)
print("Script created")

Script created


In [19]:
!python train_hybrid_simple.py

Device: cuda
Loading data...
Train: 73,295 | Test: 9,162
Loading tokenizers...
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 71.8kB/s]
vocab.json: 899kB [00:00, 111MB/s]
merges.txt: 456kB [00:00, 98.7MB/s]
special_tokens_map.json: 100% 772/772 [00:00<00:00, 3.59MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 1499.49it/s, Materializing param=pooler.dense.weight]
Loading weights: 100% 197/197 [00:00<00:00, 1323.12it/s, Materializing param=encoder.layer.11.output.dense.weight]
RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.we

In [20]:
from google.colab import files
files.download('results/hybrid_evaluation.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>